# NB_hospital_daily_summary

Builds `adb_caresync.gold.hospital_daily_summary` — one row per hospital per calendar day with test volumes, patient/doctor counts, abnormal/critical rates, and day-over-day trend. Full rebuild via `CREATE OR REPLACE TABLE`.

In [0]:
%sql
CREATE OR REPLACE TABLE adb_caresync.gold.hospital_daily_summary AS
WITH daily_agg AS (
    SELECT
        lr.hospital_id,
        lr.test_date AS summary_date,
        COUNT(*) AS total_tests,
        COUNT(DISTINCT lr.patient_id) AS unique_patients_tested,
        COUNT(DISTINCT lr.doctor_id) AS active_doctor_count,
        COUNT(CASE WHEN lr.result_status = 'Abnormal' THEN 1 END) AS abnormal_test_count,
        COUNT(CASE WHEN lr.result_status = 'Critical' THEN 1 END) AS critical_test_count
    FROM adb_caresync.silver.lab_results lr
    GROUP BY lr.hospital_id, lr.test_date
)
SELECT
    da.hospital_id,
    h.hospital_name,
    da.summary_date,
    c.day_name,
    c.is_weekend,
    da.total_tests,
    da.unique_patients_tested,
    da.active_doctor_count,
    da.abnormal_test_count,
    da.critical_test_count,
    CAST(ROUND(da.abnormal_test_count / da.total_tests * 100, 2) AS DECIMAL(5,2)) AS abnormal_rate_pct,
    CAST(ROUND(da.critical_test_count / da.total_tests * 100, 2) AS DECIMAL(5,2)) AS critical_rate_pct,
    LAG(da.total_tests) OVER (PARTITION BY da.hospital_id ORDER BY da.summary_date) AS prev_day_total_tests,
    CAST(ROUND(
        (da.total_tests - LAG(da.total_tests) OVER (PARTITION BY da.hospital_id ORDER BY da.summary_date))
        / NULLIF(LAG(da.total_tests) OVER (PARTITION BY da.hospital_id ORDER BY da.summary_date), 0) * 100,
        2
    ) AS DECIMAL(6,2)) AS test_volume_change_pct
FROM daily_agg da
LEFT JOIN adb_caresync.silver.hospitals h
    ON da.hospital_id = h.hospital_id
LEFT JOIN adb_caresync.silver.calendar c
    ON c.date_key = CAST(DATE_FORMAT(da.summary_date, 'yyyyMMdd') AS INT)

In [0]:
updated_df = spark.table("adb_caresync.gold.hospital_daily_summary")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))